In [1]:
import sidechainnet as scn
import torch
from torch.utils.data import DataLoader
import numpy as np
import requests
import textwrap
from IPython.display import display, HTML, display


/home/myoung/csc6621_final_project/.venv/lib/python3.12/site-packages/sidechainnet/structure/build_info.py:225: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [2]:
proteins_data = scn.load(casp_version=12, casp_thinning=30, scn_dir="./data/")
proteins_dataloaders = scn.load(casp_version=12, casp_thinning=30, scn_dir="./data/", with_pytorch="dataloaders")


SidechainNet was loaded from ./data/sidechainnet_casp12_30.pkl.
SidechainNet was loaded from ./data/sidechainnet_casp12_30.pkl.


In [3]:

print(type(proteins_data))

#dictionary of dataloaders for train, val, and test sets
print(type(proteins_dataloaders))

<class 'sidechainnet.dataloaders.SCNDataset.SCNDataset'>
<class 'dict'>


In [ ]:
# validation loaders for different sequence identity thresholds
# valid-10: 10% sequence identity threshold, no protein in the validation set has more than 10% sequence identity to any protein in the training set
# valid-90: 90% sequence identity threshold, no protein in the validation set has more than 90% sequence identity to any protein in the training set
# valid-10 is the most challenging validation set (least similar to the training set) 
# valid-90 is the least challenging one (similar to the training set)

proteins_dataloaders.keys()

In [ ]:
train_dataloader = proteins_dataloaders['train']


val_dataloader = proteins_dataloaders['valid-30']

test_dataloader = proteins_dataloaders['test']

In [ ]:
# input variables for each protein in the dataset
# seqs: amino acid sequence of the protein
# seqs_onehot: one-hot encoded representation of the amino acid sequence, where each amino acid is represented as a binary vector indicating its presence at each position in the sequence
# seqs_int: integer encoded representation of the amino acid sequence, where each amino acid is represented as an integer index corresponding to its position in a predefined amino acid alphabet
# evolutionary: evolutionary information (position-specific scoring matrix) for the protein sequence, describes how each position in the sequence is conserved across different species
# masks: used to pad the sequences to a fixed length since sequences can have varying lengths, indicates which positions in the sequence are valid (1) and which are padding (0)

# target variables for each protein in the dataset
# angles: 3D coordinates of the protein backbone and sidechain atoms
# coords: 3D coordinates of the protein backbone atoms (N, CA, C)
# secondary: secondary structure information for the protein sequence


# metadata variables for each protein in the dataset
# ids: unique identifier for the protein from the Protein Data Bank (PDB)
# resolution: resolution of the protein structure (in Angstroms), lower means higher quality structure
# unmodified_seq: unmodified sequence of the protein (original sequence without any modifications or padding)

batch = next(iter(train_dataloader))

# List all attributes that don't start with an underscore
columns = [attr for attr in dir(batch) if not attr.startswith('_')]
print(columns)

In [ ]:
input_variables = ['seqs', 'seqs_onehot', 'seqs_int', 'evolutionary', 'masks']
target_variables = ['angles', 'coords', 'secondary']

In [ ]:
# shape of input variables

for var in input_variables:
    print(f"{var}: {getattr(batch, var).shape}")

In [ ]:
#shape of output variables

for var in target_variables:
    print(f"{var}: {getattr(batch, var).shape}")

In [ ]:
print(batch.seqs[0])

In [ ]:
def get_protein_name_from_pdb(protein) -> str:
    # 1. Extract the PDB ID from the protein's metadata
    pdb_id = protein.id.split('_')[0]

    # 2. Ping the official Protein Data Bank API to get the biological name
    try:
        url = f"https://data.rcsb.org/rest/v1/core/entry/{pdb_id}"
        response = requests.get(url)
        response.raise_for_status() # Check for HTTP errors
        
        # Extract the title from the JSON payload
        actual_protein_name = response.json()['struct']['title']
        
        format_protein_name = textwrap.fill(actual_protein_name, width=30)
        
        
        
    except Exception as e:
        print(f"Could not fetch name from PDB API: {e}")
        format_protein_name = protein.id
        
    return format_protein_name
    

In [ ]:
def view_protein_name_from_pdb(proteins_data, index: int):
    
    protein = proteins_data[index]
    protein.fastbuild(inplace=True)

    protein_name = get_protein_name_from_pdb(protein)

    protein_view = protein.to_3Dmol()
    
    view_html = protein_view._make_html()

    display(HTML(f"""
    <div style="position:relative; display:inline-block; line-height:0;">
    <div style="position:absolute; top:10px; left:50%; transform:translateX(-50%);
                background:rgba(0,0,0,0.65); color:white; padding:5px 14px;
                border-radius:4px; font-size:14px; font-weight:bold;
                white-space:nowrap; z-index:999; line-height:normal;">
        Protein: {protein_name}
    </div>
    {view_html}
    </div>
"""))

In [ ]:
# Views the name of the protein from the PDB API for the 4th protein in the dataset (index=3)

view_protein_name_from_pdb(proteins_data, index=3)